# Week 5 - Spark Questions

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [5]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Celebal_Week5")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [19]:
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

### Dataset Exploration

In [7]:
print("Number of Rows:", df.count())
print("Number of Columns:", len(df.columns))
df.printSchema()

Number of Rows: 9994
Number of Columns: 21
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



### Data Cleaning

In [ ]:
# Remove Duplicates

df = df.dropDuplicates()
print("Rows after removing duplicates:")
print(df.count())

Rows after removing duplicates:
9994


In [ ]:
# Check Missing Values

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [ ]:
# Fill Null Values

df = df.fillna({
    "Sales":0,
    "Profit":0,
    "Discount":0
})

df.show()

+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|   222|CA-2015-169397|12/24/2015|12/27/2015|   First Class|   JB-15925|      Joni Blumstein|   Consumer|United States|       Dublin|          Ohio|      43017|   East|OFF-BI-100028

### Filtering Operations

In [ ]:
# Filter West Region

west_df = df.filter(
    col("Region")=="West"
)

west_df.show()

+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|  1142|CA-2014-146969| 9/29/2014| 10/3/2014|Standard Class|   AP-10915|      Arthur Prichep|   Consumer|United States|  Los Angeles|California|      90045|  West|FUR-FU-10004188|      Furniture| Furnis

In [ ]:
# Filter Category

tech_df = df.filter(
    col("Category")=="Technology"
)

tech_df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+
|     8|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|   Brosina Hoffman|   Consumer|United States|  Los Angeles|    California|      90032|   West|TEC-PH-10002275|Technology|      Phones|Mi

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .csv("Sample - Superstore.csv")

In [25]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### Aggregation Functions

In [26]:
df.select(
    count("*").alias("Total Records"),
    sum("Sales").alias("Total Sales"),
    avg("Sales").alias("Average Sales"),
    min("Sales").alias("Minimum Sales"),
    max("Sales").alias("Maximum Sales")
).show()

+-------------+-----------------+-----------------+-------------+-------------+
|Total Records|      Total Sales|    Average Sales|Minimum Sales|Maximum Sales|
+-------------+-----------------+-----------------+-------------+-------------+
|         9994|2297200.860299955|229.8580008304938|        0.444|     22638.48|
+-------------+-----------------+-----------------+-------------+-------------+



### GroupBy Operations

In [27]:
df.groupBy("Category") \
    .agg(
        sum("Sales").alias("Total Sales"),
        avg("Sales").alias("Average Sales")
    ) \
    .show()

+---------------+-----------------+------------------+
|       Category|      Total Sales|     Average Sales|
+---------------+-----------------+------------------+
|Office Supplies|719047.0320000029|119.32410089611732|
|      Furniture|741999.7952999998|349.83488698727007|
|     Technology|836154.0329999966|452.70927612344155|
+---------------+-----------------+------------------+



In [28]:
df.groupBy("Region") \
    .count() \
    .show()

+-------+-----+
| Region|count|
+-------+-----+
|  South| 1620|
|Central| 2323|
|   East| 2848|
|   West| 3203|
+-------+-----+



### HAVING Condition

In [29]:
df.groupBy("City") \
    .count() \
    .filter(col("count") > 100) \
    .show()

+-------------+-----+
|         City|count|
+-------------+-----+
|  Springfield|  163|
|       Dallas|  157|
| Philadelphia|  537|
|  Los Angeles|  747|
|San Francisco|  510|
|    San Diego|  170|
|      Detroit|  115|
|     Columbus|  222|
|      Chicago|  314|
|      Seattle|  428|
|New York City|  915|
|      Houston|  377|
| Jacksonville|  125|
+-------------+-----+



### Schema Modifications

In [ ]:
# Rename Column

df = df.withColumnRenamed(
    "Customer Name",
    "Customer_Name"
)

df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer_Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

In [ ]:
# Type Casting

df = df.withColumn(
    "Order Date",
    to_date(
        col("Order Date"),
        "M/d/yyyy"
    )
)

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### Remove Invalid Records

In [33]:
clean_df = df.filter(
    col("Customer_Name").isNotNull()
)

clean_df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer_Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

### Complete Data Processing Pipeline

In [34]:
final_df = (
    df
    .dropDuplicates()
    .fillna({"Sales":0})
    .groupBy("Category")
    .agg(
        sum("Sales").alias("Total Revenue")
    )
)

final_df.show()

+---------------+-----------------+
|       Category|    Total Revenue|
+---------------+-----------------+
|Office Supplies| 719047.032000001|
|      Furniture|741999.7953000015|
|     Technology|836154.0329999998|
+---------------+-----------------+

